In [ ]:
# Colab hosted GPU. Chạy cell 2 để đăng nhập bằng liên kết Google.
%pip -q install zstandard pexpect
!command -v rclone >/dev/null || (apt-get -qq update && apt-get -qq install -y rclone)


In [ ]:
from pathlib import Path
import os, json, math, random, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix)
import subprocess, shutil, time, getpass, pexpect
TARGET_SYMBOL = "VIAC"
ACCOUNT = input("Google account for this dataset: ").strip()
FOLDER_ID = input("Google Drive data folder ID: ").strip()
DATA = Path("/content/ai4life_input")
OUT = Path("/content/ai4life_output")
DATA.mkdir(exist_ok=True)
for exe in ("gcloud","rclone"):
    if not shutil.which(exe): raise RuntimeError(f"Missing executable: {exe}")

# Use Google's interactive CLI authorization, independently of Colab DriveFS.
child = pexpect.spawn("gcloud", ["auth","login",ACCOUNT,"--enable-gdrive-access","--no-launch-browser"],
                      encoding="utf-8", timeout=300)
try:
    while True:
        event = child.expect([r"https://accounts\.google\.com/[^\s]+",
                              r"(?i)enter (?:the )?(?:verification|authorization) code[^:]*:",
                              r"(?i)\(Y/n\)\?", pexpect.EOF])
        if event == 0:
            print("Mở liên kết Google bên dưới bằng tài khoản sở hữu dữ liệu:",flush=True)
            print(child.after,flush=True)
        elif event == 1:
            child.sendline(getpass.getpass("Dán mã Google vào ô ẩn này: ").strip())
        elif event == 2:
            child.sendline("y")
        else: break
finally:
    child.close()
if child.exitstatus != 0:
    raise RuntimeError("Google chưa hoàn tất đăng nhập. Chạy lại cell này.")

# Token is supplied to the child process through its environment, never printed.
def transfer(*args, folder_id=None, capture=False):
    for attempt in range(2):
        token = subprocess.check_output(["gcloud","auth","print-access-token",
                                         "--account",ACCOUNT],text=True).strip()
        env = os.environ.copy()
        env.update(RCLONE_CONFIG_AIDRIVE_TYPE="drive",
                   RCLONE_CONFIG_AIDRIVE_TOKEN=json.dumps({"access_token":token,"token_type":"Bearer"}))
        if folder_id: env["RCLONE_CONFIG_AIDRIVE_ROOT_FOLDER_ID"]=folder_id
        res=subprocess.run(["rclone",*args],env=env,text=True,
                           stdout=subprocess.PIPE if capture else None)
        if res.returncode==0: return res.stdout
    raise RuntimeError("Drive API transfer failed. See rclone output above.")

# Only job descriptions are copied. The compressed data stays on Drive.
transfer("copy","aidrive:",str(DATA),"--include","metadata.json","--include","manifest.json",
         "--include","reviewed_test_windows.csv",folder_id=FOLDER_ID)
inventory=json.loads(transfer("lsjson","aidrive:","--recursive","--files-only",
    "--include","*.dbn.zst",folder_id=FOLDER_ID,capture=True))
remote_sizes={item["Path"]: item["Size"] for item in inventory}
if not remote_sizes: raise FileNotFoundError("No DBN files found in the selected Drive folder.")
print(f"{len(remote_sizes)} compressed files available for streaming.")
REMOTE_OUT="aidrive:AI4LIFE_DHGNN_RESULTS"
def backup(folder):
    remote=REMOTE_OUT+"/"+folder.relative_to(OUT).as_posix()
    transfer("copy",str(folder),remote,"--exclude","*.tmp","--exclude","*.partial")
OUT.mkdir(parents=True, exist_ok=True)
SEED, NODES, HIDDEN, BATCH, EPOCHS, PATIENCE = 42, 32, 48, 48, 15, 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(DATA, DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")


In [ ]:
"""Prepare causal per-session tensors from local Databento XNAS.ITCH DBN v1."""
from collections import Counter
import zstandard as zstd
from datetime import date, datetime, timedelta, timezone
import json
from pathlib import Path
import struct
import time

import numpy as np

NS = 10**9
MBO = struct.Struct('<BBHIQQqIBBccQiI')
BAR = struct.Struct('<BBHIQqqqqQ')
SYMBOLS = {TARGET_SYMBOL}
SHORT = {date(2020, 11, 27), date(2020, 12, 24)}
HOLIDAYS = {'2020-11-26', '2020-12-25', '2021-01-01', '2021-01-18',
            '2021-02-15', '2021-04-02'}
NODE = ['side', 'distance_from_mid_bps', 'remaining_shares', 'observed_age_seconds',
        'birth_known', 'initial_shares', 'executed_shares', 'cancelled_shares']
FLOW = ['add_count', 'add_qty', 'cancel_count', 'cancel_qty', 'replace_count',
        'replace_removed_qty', 'fill_count', 'fill_qty', 'trade_count', 'trade_qty']
BOOK = ['mid_usd', 'spread_bps', 'bid_depth_5', 'ask_depth_5', 'depth_imbalance_5']


from contextlib import contextmanager

@contextmanager
def stream_dbn(path):
    remote_path = Path(path).relative_to(DATA).as_posix()
    token = subprocess.check_output(["gcloud","auth","print-access-token","--account",ACCOUNT],
                                    text=True).strip()
    env = os.environ.copy()
    env.update(RCLONE_CONFIG_AIDRIVE_TYPE="drive",
               RCLONE_CONFIG_AIDRIVE_ROOT_FOLDER_ID=FOLDER_ID,
               RCLONE_CONFIG_AIDRIVE_TOKEN=json.dumps({"access_token":token,"token_type":"Bearer"}))
    proc = subprocess.Popen(["rclone","cat","aidrive:"+remote_path],stdout=subprocess.PIPE,env=env)
    try:
        with zstd.ZstdDecompressor().stream_reader(proc.stdout, read_across_frames=True,
                                                    closefd=False) as decoded:
            yield decoded
        if proc.wait() != 0:
            raise IOError(f"Drive stream interrupted: {remote_path}. Rerun the preprocessing cell.")
    finally:
        proc.stdout.close()
        if proc.poll() is None:
            proc.terminate()
        proc.wait()

def dbn_records(path, layout, rtype):
    with stream_dbn(path) as source:
        head = source.read(8)
        if head[:4] != b'DBN\x01':
            raise ValueError(f'Expected DBN v1: {path}')
        header_size = int.from_bytes(head[4:], 'little')
        if len(source.read(header_size)) != header_size:
            raise ValueError(f'Truncated DBN header: {path}')
        remainder = b''
        while block := source.read(4 * 1024 * 1024):
            block = remainder + block
            end = len(block) // layout.size * layout.size
            for row in layout.iter_unpack(memoryview(block)[:end]):
                if row[:2] != (layout.size // 4, rtype):
                    raise ValueError(f'Unexpected DBN record: {path}')
                yield row
            remainder = block[end:]
        if remainder:
            raise ValueError(f'Truncated DBN record: {path}')


def session_bounds(day):
    # Historical New York offsets for the supported October 2020–April 2021 interval.
    offset = 4 if day < date(2020, 11, 1) or day >= date(2021, 3, 14) else 5
    midnight = datetime.combine(day, datetime.min.time(), timezone.utc)
    start = int((midnight + timedelta(hours=9 + offset, minutes=30)).timestamp())
    minutes = 210 if day in SHORT else 390
    return start * NS, minutes


class OrderBook:
    def __init__(self, audit):
        self.audit = audit
        self.clear()

    def clear(self):
        self.orders = {}
        self.levels = {b'B': {}, b'A': {}}

    def add(self, oid, price, qty, side, birth):
        if side not in self.levels or qty <= 0 or price <= 0 or price >= 2**62:
            self.audit['invalid_add'] += 1
            return
        if oid in self.orders:
            self.audit['duplicate_add'] += 1
            self.remove(oid, self.orders[oid][0], None)
        self.orders[oid] = [qty, side, price, birth, qty if birth is not None else None, 0, 0]
        self.levels[side].setdefault(price, {})[oid] = None

    def remove(self, oid, qty, kind):
        order = self.orders.get(oid)
        if order is None:
            self.audit['unmatched_reduction'] += 1
            return
        if qty > order[0]:
            self.audit['over_reduction'] += 1
        applied = min(qty, order[0])
        order[0] -= applied
        if kind in (5, 6):
            order[kind] += applied
        if order[0] == 0:
            bucket = self.levels[order[1]][order[2]]
            del bucket[oid]
            if not bucket:
                del self.levels[order[1]][order[2]]
            del self.orders[oid]

    def snapshot(self, t, x, ids, groups):
        bids = sorted(self.levels[b'B'], reverse=True)
        asks = sorted(self.levels[b'A'])
        valid = bool(bids and asks and bids[0] < asks[0])
        mid = (bids[0] + asks[0]) / 2 if valid else np.nan
        depths = []
        for half, (side, prices) in enumerate(((b'B', bids), (b'A', asks))):
            levels = self.levels[side]
            depths.append(sum(self.orders[oid][0] for p in prices[:5] for oid in levels[p]))
            slot = half * (len(x) // 2)
            stop = slot + len(x) // 2
            for rank, price in enumerate(prices):
                for oid in levels[price]:
                    qty, _, _, birth, initial, filled, cancelled = self.orders[oid]
                    age = (t - birth) / NS if birth is not None else np.nan
                    x[slot] = [1 if half == 0 else -1, (price / mid - 1) * 1e4,
                               qty, age, birth is not None, initial if initial is not None else np.nan,
                               filled, cancelled]
                    ids[slot] = oid
                    # Each column defines one hyperedge family. IDs are local to this second.
                    groups[slot] = [half, half * 3 + min(rank // 2, 2),
                                    half * 5 + (4 if birth is None else
                                                0 if age < 1 else 1 if age < 5 else 2 if age < 30 else 3)]
                    slot += 1
                    if slot == stop:
                        break
                if slot == stop:
                    break
        bid, ask = depths
        spread = (asks[0] - bids[0]) / mid * 1e4 if valid else np.nan
        return [mid / NS, spread, bid, ask, (bid - ask) / (bid + ask) if bid + ask else np.nan], valid


def read_context(path):
    bars = {}
    for r in dbn_records(path, BAR, 33):
        ts, op, hi, lo, cl, volume = r[4:]
        if ts in bars or not (0 < lo <= min(op, cl) <= max(op, cl) <= hi):
            raise ValueError(f"Invalid or duplicate bar in {path} at {ts}")
        bars[ts] = (1e4 * math.log(cl / op), 1e4 * math.log(hi / lo), math.log1p(volume))
    return bars


def preprocess(path, symbol, day, context, nodes, output):
    started = time.perf_counter()
    start, minutes = session_bounds(day)
    seconds = minutes * 60
    end = start + seconds * NS
    x = np.full((seconds, nodes, len(NODE)), np.nan, np.float32)
    ids = np.zeros((seconds, nodes), np.uint64)
    groups = np.full((seconds, nodes, 3), -1, np.int8)
    flow = np.zeros((seconds, len(FLOW)), np.int64)
    book_values = np.full((seconds, len(BOOK)), np.nan, np.float32)
    valid = np.zeros(seconds, bool)
    audit = Counter()
    book = OrderBook(audit)
    step = 0
    previous = None
    last_timestamp = -1
    first_event = None
    last_event = None
    raw_trade_qty = 0
    records = iter(dbn_records(path, MBO, 160))
    current = next(records, None)
    while current is not None:
        following = next(records, None)
        _, _, _, _, event, oid, price, qty, flags, channel, action, side, recv, _, seq = current
        synthetic = bool(flags & 32)
        stamp = recv if synthetic else event
        if stamp < last_timestamp:
            audit['timestamp_regressions'] += 1
            stamp = last_timestamp
        last_timestamp = stamp
        while step < seconds and start + (step + 1) * NS <= stamp:
            cutoff = start + (step + 1) * NS
            book_values[step], valid[step] = book.snapshot(cutoff, x[step], ids[step], groups[step])
            step += 1
        if stamp >= end:
            for _ in records:
                pass
            break
        in_session = stamp >= start and not synthetic
        if in_session:
            first_event = stamp if first_event is None else first_event
            last_event = stamp
            audit['regular_records'] += 1
        def match(other, expected_action, same_order=False):
            return (other is not None and other[10] == expected_action and
                    (other[4], other[9], other[14]) == (event, channel, seq) and
                    (not same_order or other[5] == oid))
        offset = None
        if action == b'R':
            book.clear()
            audit['snapshot_resets' if synthetic else 'source_resets'] += 1
        elif action == b'A':
            replaced = match(previous, b'C')
            book.add(oid, price, qty, side, None if synthetic else stamp)
            if not replaced and not synthetic:
                offset = 0
        elif action == b'C':
            if match(previous, b'F', True):
                book.remove(oid, qty, 5)
                offset = 6
            elif match(following, b'A'):
                book.remove(oid, qty, None)
                offset = 4
            else:
                book.remove(oid, qty, 6)
                offset = 2
        elif action == b'T' and 0 < price < 2**62:
            offset = 8
            if in_session:
                raw_trade_qty += qty
        elif action != b'F':
            audit['unexpected_action_or_price'] += 1
        if in_session and offset is not None:
            flow[step, offset] += 1
            flow[step, offset + 1] += qty
        previous, current = current, following
    while step < seconds:
        cutoff = start + (step + 1) * NS
        book_values[step], valid[step] = book.snapshot(cutoff, x[step], ids[step], groups[step])
        step += 1
    if first_event is None:
        raise ValueError(f'No regular-session records: {path}')
    assert int(flow[:, 9].sum()) == raw_trade_qty
    # A DBN bar is keyed by its start. The last completed bar starts 60 seconds before cutoff.
    market = np.full((minutes, 2, 3), np.nan, np.float64)
    for minute in range(minutes):
        bar_start = start + minute * 60 * NS
        for column, etf in enumerate(('SPY', 'QQQ')):
            if bar_start in context[etf]:
                market[minute, column] = context[etf][bar_start]
    output.mkdir(parents=True, exist_ok=True)
    dest = output / f'{day}_{symbol}.npz'
    temporary = dest.with_suffix('.tmp')
    with temporary.open('wb') as stream:
        np.savez_compressed(stream, x=x.reshape(minutes, 60, nodes, len(NODE)),
                            order_id=ids.reshape(minutes, 60, nodes),
                            mask=(ids != 0).reshape(minutes, 60, nodes),
                            edge_group=groups.reshape(minutes, 60, nodes, 3),
                            flow=flow.reshape(minutes, 60, len(FLOW)),
                            flow_minute=flow.reshape(minutes, 60, len(FLOW)).sum(axis=1),
                            book=book_values.reshape(minutes, 60, len(BOOK)),
                            book_valid=valid.reshape(minutes, 60), market=market,
                            cutoff_ns=start + np.arange(1, minutes + 1, dtype=np.int64) * 60 * NS,
                            node_columns=NODE, flow_columns=FLOW, book_columns=BOOK)
    temporary.replace(dest)
    report = dict(symbol=symbol, day=str(day), minutes=minutes, audit=dict(audit),
                  trade_qty=raw_trade_qty, valid_book_seconds=int(valid.sum()),
                  first_event_utc=datetime.fromtimestamp(first_event / NS, timezone.utc).isoformat(),
                  last_event_utc=datetime.fromtimestamp(last_event / NS, timezone.utc).isoformat(),
                  market_missing=int(np.isnan(market[:, :, 0]).sum()),
                  bytes=dest.stat().st_size, seconds=round(time.perf_counter() - started, 2))
    dest.with_suffix('.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    return report


In [ ]:
# Lập danh mục theo metadata, kiểm tra kích thước theo manifest trước khi xử lý.
jobs, manifests = {}, {}
wanted = {("mbo", s) for s in SYMBOLS} | {("ohlcv-1m", s) for s in ("SPY","QQQ")}
for p in DATA.rglob("metadata.json"):
    q = json.loads(p.read_text(encoding="utf-8-sig")).get("query", {})
    syms = q.get("symbols", [])
    if len(syms) != 1 or q.get("dataset") != "XNAS.ITCH": continue
    key = q["schema"], syms[0]
    if key not in wanted: continue
    if key in jobs: raise ValueError(f"Job trùng {key}")
    jobs[key] = p.parent
    files = json.loads((p.parent/"manifest.json").read_text(encoding="utf-8-sig"))["files"]
    manifests[key] = {f["filename"]: f for f in files}
assert set(jobs) == wanted, wanted - jobs.keys()

def source(key, name):
    p = jobs[key] / name
    if remote_sizes.get(p.relative_to(DATA).as_posix()) != manifests[key][name]["size"]: raise IOError(p)
    return p

def split_of(d):
    if d < date(2020,11,1): return "reference"
    if d < date(2021,2,1): return "train"
    if d <= date(2021,2,12): return "validation"
    if d < date(2021,3,1): return "calibration"
    if d < date(2021,4,1): return "case_study"
    return "test"

days = [t.date() for t in pd.date_range("2020-10-01","2021-04-30",freq="B")
        if str(t.date()) not in HOLIDAYS]
assert len(days) == 146
context = {}
for s in ("SPY","QQQ"):
    key = "ohlcv-1m", s
    name, = [n for n in manifests[key] if n.endswith(".dbn.zst")]
    context[s] = read_context(source(key, name))

# Cache tách theo phiên bản đặc trưng, số node và nội dung manifest nguồn.
cache_id = hashlib.sha256(repr(("v1", NODES, sorted((k,n,v["hash"]) for k,files in manifests.items()
                                     for n,v in files.items()))).encode()).hexdigest()[:12]
CACHE = OUT / cache_id / "sessions"
RES = OUT / cache_id / "results"
for folder in (CACHE, RES): folder.mkdir(parents=True, exist_ok=True)
rows = []
for day in days:
    for symbol in sorted(SYMBOLS):
        p = source(("mbo",symbol), f"xnas-itch-{day:%Y%m%d}.mbo.dbn.zst")
        dst = CACHE / f"{day}_{symbol}.npz"
        rows.append(dict(day=str(day), symbol=symbol, split=split_of(day), source=str(p), path=str(dst)))
index = pd.DataFrame(rows)
assert set(index.symbol) == {TARGET_SYMBOL} and len(index) == len(days)
index.to_csv(RES/"split_index.csv",index=False)
display(index.groupby("split",sort=False).agg(first=("day","min"), last=("day","max"), sessions=("day","nunique")))

# Resume from previous runs by restoring the session cache for this manifest.
remote_cache = REMOTE_OUT+"/"+CACHE.relative_to(OUT).as_posix()
transfer("mkdir",remote_cache)
transfer("copy",remote_cache,str(CACHE))
backup(RES)


In [ ]:
# Đọc luồng trực tiếp từ Google Drive. Gián đoạn giữa phiên: chạy lại cell để tiếp tục.
audit_rows = []
for i,row in enumerate(index.itertuples(index=False),1):
    p = Path(row.path)
    report_path = p.with_suffix(".json")
    try:
        report = json.loads(report_path.read_text()) if report_path.exists() else {}
    except (ValueError, OSError):
        report = {}
    if not p.exists() or report.get("bytes") != p.stat().st_size:
        report = preprocess(Path(row.source), row.symbol, date.fromisoformat(row.day), context, NODES, CACHE)
    audit_rows.append(report)
    if i % 64 == 0 or i == len(index):
        backup(CACHE)
        print(f"{i}/{len(index)}", flush=True)
audit = pd.json_normalize(audit_rows)
audit.to_csv(RES/"session_audit.csv", index=False)
errors = ["duplicate_add","invalid_add","unmatched_reduction","over_reduction","timestamp_regressions","unexpected_action_or_price"]
bad = audit.reindex(columns=["audit."+k for k in errors]).fillna(0).gt(0).any(axis=1)
display(audit.loc[bad, ["day","symbol"]])
index = index.merge(audit[["day","symbol"]].assign(session_ok=~bad), on=["day","symbol"],validate="one_to_one")
index = index[index.session_ok].reset_index(drop=True)
print("Mã–phiên đủ điều kiện:",len(index))

backup(RES)


In [ ]:
# EDA trên reference/train. Mỗi điểm trên đồ thị là trung vị cùng phút qua nhiều phiên.
profiles, daily = [], []
for r in index[index.split.isin(["reference","train"])].itertuples(index=False):
    with np.load(r.path) as d:
        flow = d["flow_minute"]
        book = d["book"]
        good = d["book_valid"].all(1) & np.isfinite(d["market"]).all(axis=(1,2))
        daily.append([r.day,r.symbol,flow[:,9].sum(),good.sum(),len(good)])
        if len(flow) == 390:
            profiles.append(pd.DataFrame(dict(symbol=r.symbol, minute=np.flatnonzero(good),
                  trade_volume=flow[good,9], cancel_count=flow[good,2],
                  spread_bps=book[good,:,1].mean(1))))
daily = pd.DataFrame(daily,columns=["day","symbol","volume","usable_minutes","minutes"])
daily.to_csv(RES/"eda_daily.csv",index=False)
display(daily.groupby("symbol").agg({"volume":"median","usable_minutes":"sum","minutes":"sum"}))
profile = pd.concat(profiles).groupby(["minute","symbol"]).median()
fig, axes = plt.subplots(3,1,figsize=(13,10),sharex=True)
for ax, name in zip(axes,profile.columns):
    profile[name].unstack().rolling(5,min_periods=1).mean().plot(ax=ax,legend=ax is axes[0])
    ax.set_ylabel(name); ax.grid(alpha=.2)
axes[-1].set_xlabel("Phút từ 09:30 New York")
fig.tight_layout(); fig.savefig(RES/"intraday.png",dpi=160); plt.show()

backup(RES)


In [ ]:
# Từ cache 1 giây, tạo 6 snapshot cho mỗi phút; giữ tổng flow của đủ 60 giây.
def session_arrays(row):
    with np.load(row.path) as d:
        good = d["book_valid"].all(1) & np.isfinite(d["market"]).all(axis=(1,2))
        count = len(good)
        ix = np.flatnonzero(good)
        x = d["x"][ix,9::10].copy()
        for k in (2,3,5,6,7): x[...,k] = np.log1p(np.maximum(x[...,k],0))
        x = np.nan_to_num(x)
        mask = d["mask"][ix,9::10]
        edge = d["edge_group"][ix,9::10]
        book = d["book"][ix,9::10,1:].copy()
        book[...,1:3] = np.log1p(book[...,1:3])
        f = d["flow"][ix].reshape(len(ix),6,10,len(FLOW)).sum(2)
        step = np.concatenate([np.log1p(f),book],-1).astype("float32")
        ctx = np.column_stack([d["market"][ix].reshape(len(ix),6),
              np.sin(2*np.pi*ix/390),np.cos(2*np.pi*ix/390),
              np.full(len(ix),count==210)]).astype("float32")
        return dict(x=x,mask=mask,edge=edge,step=step,ctx=ctx), d["cutoff_ns"][ix], ix

rng = np.random.default_rng(SEED)
parts = {k:[] for k in ("x","step","ctx")}
for row in index[index.split=="train"].itertuples(index=False):
    a,_,ix = session_arrays(row)
    if not len(ix): continue
    take = rng.choice(len(ix),min(12,len(ix)),replace=False)
    nx = a["x"][take][a["mask"][take]]
    parts["x"].append(nx[rng.choice(len(nx),min(len(nx),256),replace=False)])
    parts["step"].append(a["step"][take].reshape(-1,a["step"].shape[-1]))
    parts["ctx"].append(a["ctx"][take])
stats = {}
for k,chunks in parts.items():
    v = np.concatenate(chunks).astype("float64")
    mu,sd = v.mean(0),v.std(0)
    stats[k] = mu.astype("float32"), np.where(sd>1e-5,sd,1).astype("float32")
np.savez(RES/"train_normalization.npz",**{f"{k}_{n}":v[i] for k,v in stats.items()
                                         for i,n in enumerate(("mean","scale"))})
del parts

def batches(split, epoch=0):
    rng = np.random.default_rng(SEED+epoch)
    rows = list(index[index.split==split].itertuples(index=False))
    if split=="train": rng.shuffle(rows)
    for row in rows:
        a,ts,minute = session_arrays(row)
        for k,(mu,sd) in stats.items(): a[k] = np.clip((a[k]-mu)/sd,-10,10)
        order = rng.permutation(len(ts)) if split=="train" else np.arange(len(ts))
        for start in range(0,len(ts),BATCH):
            sl = order[start:start+BATCH]
            yield ({k:torch.from_numpy(v[sl]).to(DEVICE) for k,v in a.items()},
                   row.symbol,row.day,ts[sl],minute[sl])

backup(RES)


In [ ]:
class HyperLayer(nn.Module):
    def __init__(self, hidden, dynamic):
        super().__init__()
        self.dynamic = dynamic
        self.edge_mlp = nn.Sequential(nn.Linear(hidden,hidden), nn.GELU(),
                                      nn.Linear(hidden,hidden))
        self.node_mlp = nn.Sequential(nn.Linear(hidden*2,hidden), nn.GELU(),
                                      nn.Linear(hidden,hidden))
        self.norm = nn.LayerNorm(hidden)

    def forward(self, h, mask, incidence):
        a = incidence
        if self.dynamic:
            # Hard kNN is rebuilt for each snapshot and layer. Gradients pass
            # through messages, NOT the discrete neighbor-selection operation.
            with torch.no_grad():
                dist = torch.cdist(h.detach(), h.detach())
                dist = dist.masked_fill(~mask[:,None,:], 1e9)
                # Include all near-ties at the kth distance. Arbitrarily picking
                # tied indices can make node relabeling change the prediction.
                cutoff = dist.topk(min(6,h.shape[1]),dim=-1,largest=False).values[:,:,-1:]
                tolerance = 1e-5 * cutoff.clamp_min(1.)
                knn = (dist <= cutoff + tolerance).to(h.dtype)
                knn = knn * mask[:,:,None] * mask[:,None,:]
                a = torch.cat([a,knn.transpose(1,2)],dim=2)
        degree = a.sum(1).clamp_min(1.)
        edge = torch.bmm(a.transpose(1,2),h) / degree[:,:,None]
        edge = self.edge_mlp(edge)
        back = torch.bmm(a,edge) / a.sum(2).clamp_min(1.)[:,:,None]
        h = self.norm(h + self.node_mlp(torch.cat([h,back],-1)))
        return h * mask[:,:,None]

class DHGNNEnergy(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Sequential(nn.Linear(len(NODE),HIDDEN),nn.GELU())
        self.hg = nn.ModuleList([HyperLayer(HIDDEN,True) for _ in range(2)])
        self.gru = nn.GRU(2*HIDDEN+len(FLOW)+4,HIDDEN,batch_first=True)
        self.head = nn.Sequential(nn.Linear(HIDDEN+9,HIDDEN),nn.GELU(),nn.Linear(HIDDEN,1))
    def forward(self,b):
        B,T,N,D = b["x"].shape
        mask = b["mask"].reshape(-1,N)
        group = b["edge"].reshape(-1,N,3).long()
        edges = torch.cat([F.one_hot(group[...,i].clamp_min(0),w)
                           for i,w in enumerate((2,6,10))],-1).float()*mask.unsqueeze(-1)
        h = self.embed(b["x"].reshape(-1,N,D))*mask.unsqueeze(-1)
        for layer in self.hg: h=layer(h,mask,edges)
        mean = h.sum(1)/mask.sum(1,keepdim=True).clamp_min(1)
        maximum = h.masked_fill(~mask.unsqueeze(-1),-1e6).amax(1)
        maximum = torch.where(mask.any(1,keepdim=True),maximum,0.)
        pooled = torch.cat([mean,maximum],-1).reshape(B,T,-1)
        _,state = self.gru(torch.cat([pooled,b["step"]],-1))
        return self.head(torch.cat([state[-1],b["ctx"]],-1)).flatten()

# Biến thể đối chiếu thay đổi độ đồng bộ giữa đồ thị lệnh và dòng sự kiện.
# Dịch cả node, mask và hyperedge cùng nhau, chỉ trong cùng cửa sổ.
def perturb(b,rng,mild=False):
    B,T = b["x"].shape[:2]
    shift = np.ones(B,dtype=int) if mild else rng.integers(1,T,size=B)
    order = (torch.arange(T,device=DEVICE)[None,:]+torch.as_tensor(shift,device=DEVICE)[:,None])%T
    idx = torch.arange(B,device=DEVICE)[:,None]
    return {**b, **{k:b[k][idx,order] for k in ("x","mask","edge")}}

model = DHGNNEnergy().to(DEVICE)
opt = torch.optim.AdamW(model.parameters(),lr=5e-4,weight_decay=1e-4)
def epoch_pass(split,epoch):
    training = split=="train"
    model.train(training)
    rng = np.random.default_rng(SEED+epoch if training else 9107)
    total = hits = count = 0
    with torch.set_grad_enabled(training):
        for b,*_ in batches(split,epoch):
            observed, altered = model(b),model(perturb(b,rng))
            loss = F.softplus(1+observed-altered).mean()+0.001*(observed.square()+altered.square()).mean()
            if training:
                opt.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(),2); opt.step()
            total += loss.item()*len(observed)
            hits += (altered>observed).sum().item()
            count += len(observed)
    if not count: raise ValueError(f"Tập rỗng: {split}")
    return total/count,hits/count

history, best, stale = [], float("inf"), 0
checkpoint = RES/"best_dhgnn_ebm.pt"
for epoch in range(1,EPOCHS+1):
    tr,ta = epoch_pass("train",epoch)
    va,pa = epoch_pass("validation",0)
    if not np.isfinite([tr,va]).all(): raise FloatingPointError("Loss không hữu hạn")
    history.append(dict(epoch=epoch,train_loss=tr,val_loss=va,train_pair_acc=ta,val_pair_acc=pa))
    pd.DataFrame(history).to_csv(RES/"training_history.csv",index=False)
    print(epoch,history[-1],flush=True)
    if va < best:
        best,stale = va,0
        tmp=checkpoint.with_suffix(".tmp")
        torch.save(dict(model=model.state_dict(),epoch=epoch,hidden=HIDDEN,nodes=NODES,
                        config_id=cache_id),tmp); os.replace(tmp,checkpoint)
        backup(RES)
    else:
        stale+=1
        if stale>=PATIENCE: break
model.load_state_dict(torch.load(checkpoint,map_location=DEVICE,weights_only=True)["model"])
pd.DataFrame(history).plot(x="epoch",y=["train_loss","val_loss"],ylabel="Contrastive loss")
plt.tight_layout(); plt.savefig(RES/"training_loss.png",dpi=160); plt.show()

backup(RES)


In [ ]:
# Hiệu chỉnh ngưỡng bằng tháng 2. Tháng 3 dành cho case study, tháng 4 là test.
@torch.inference_mode()
def score_split(split,synthetic=False):
    model.eval()
    rng = np.random.default_rng(2027)
    frames=[]
    for b,s,day,ts,minute in batches(split):
        f=pd.DataFrame(dict(symbol=s,day=day,window_end_utc=pd.to_datetime(ts,utc=True),
                            minute=minute,bucket=minute//30,energy=model(b).cpu().numpy()))
        if synthetic:
            for name in ("full","mild"):
                f[name]=model(perturb(b,rng,name=="mild")).cpu().numpy()
        frames.append(f)
    if not frames: raise ValueError(f"Tập rỗng: {split}")
    return pd.concat(frames,ignore_index=True)

cal = score_split("calibration")
q=.99
pooled=cal.energy.quantile(q,interpolation="higher")
per_symbol=cal.groupby("symbol").energy.quantile(q,interpolation="higher")
g=cal.groupby(["symbol","bucket"]).energy
cuts=g.quantile(q,interpolation="higher").rename("threshold").to_frame()
cuts["n"]=g.size()
cuts.loc[cuts.n<100,"threshold"]=np.nan
cuts.to_csv(RES/"thresholds.csv")
per_symbol.to_csv(RES/"symbol_thresholds.csv")
(RES/"global_threshold.json").write_text(json.dumps(dict(quantile=q,threshold=float(pooled))))
def flag(df):
    df=df.join(cuts,on=["symbol","bucket"])
    df["threshold"]=df.threshold.fillna(df.symbol.map(per_symbol)).fillna(pooled)
    df["margin"]=df.energy-df.threshold
    df["alert"]=(df.margin>0).astype(int)
    return df
case=flag(score_split("case_study"))
test=flag(score_split("test",True))
case.to_csv(RES/"march_case_study.csv",index=False)
test.to_csv(RES/"april_test_scores.csv",index=False)
display(test.groupby("symbol").alert.agg(["count","sum","mean"]))

# Gộp các phút cảnh báo liên tục trong cùng phiên.
a=test.loc[test.alert==1].sort_values(["symbol","day","window_end_utc"]).copy()
keys=["symbol","day"]
a["new"]=a.groupby(keys).window_end_utc.diff().ne(pd.Timedelta(minutes=1))
a["event"]=a.groupby(keys)["new"].cumsum()
events=a.groupby(keys+["event"]).agg(start_utc=("window_end_utc","min"),
    end_utc=("window_end_utc","max"),minutes=("energy","size"),peak_margin=("margin","max")).reset_index()
events["start_utc"]-=pd.Timedelta(minutes=1)
events.to_csv(RES/"april_alert_events.csv",index=False)

ex=test[test.symbol==TARGET_SYMBOL]
if not ex.empty:
    ex=ex[ex.day==ex.day.min()]
    ax=ex.plot(x="minute",y=["energy","threshold"],figsize=(12,4),title=f"{TARGET_SYMBOL} · {ex.day.iloc[0]}")
    hits=ex[ex.alert==1]
    ax.scatter(hits.minute,hits.energy,c="red",s=14)
    ax.set_xlabel("Phút kể từ mở cửa"); plt.tight_layout()
    plt.savefig(RES/"test_energy.png",dpi=160); plt.show()

backup(RES)


In [ ]:
# Accuracy, precision, recall, F1, ROC-AUC và average precision.
# Accuracy tổng hợp đo khả năng phân biệt mẫu quan sát với biến thể tạo bằng code.
# Accuracy thực tế dùng các nhãn rà soát độc lập nếu đã có.
def metrics(y,pred,score):
    y=np.asarray(y,dtype=int)
    both=len(np.unique(y))==2
    return dict(n=len(y),positive=int(y.sum()),accuracy=accuracy_score(y,pred),
        balanced_accuracy=balanced_accuracy_score(y,pred) if both else np.nan,
        precision=precision_score(y,pred,zero_division=0),recall=recall_score(y,pred,zero_division=0),
        f1=f1_score(y,pred,zero_division=0),roc_auc=roc_auc_score(y,score) if both else np.nan,
        average_precision=average_precision_score(y,score) if both else np.nan)

results=[]
for mode in ("full","mild"):
    obs=test.margin.to_numpy()
    alt=(test[mode]-test.threshold).to_numpy()
    y=np.r_[np.zeros(len(obs),int),np.ones(len(alt),int)]
    score=np.r_[obs,alt]
    results.append(dict(task="synthetic_"+mode,paired_accuracy=np.mean(alt>obs),
                        **metrics(y,score>0,score)))
    if mode=="full":
        cm=confusion_matrix(y,score>0,labels=[0,1])
        display(pd.DataFrame(cm,index=["Observed","Synthetic"],columns=["Below threshold","Alert"]))
proxy=pd.DataFrame(results)
display(proxy.round(4)); proxy.to_csv(RES/"synthetic_metrics.csv",index=False)

# CSV tuỳ chọn: symbol,window_end_utc,label
# label=1 nếu người rà soát đánh giá cửa sổ đáng kiểm tra, label=0 nếu đã rà soát và loại.
labels_file=DATA/"reviewed_test_windows.csv"
if labels_file.exists():
    labels=pd.read_csv(labels_file)
    labels["symbol"]=labels.symbol.str.strip().str.upper()
    labels["window_end_utc"]=pd.to_datetime(labels.window_end_utc,utc=True,errors="raise")
    keys=["symbol","window_end_utc"]
    assert labels.label.isin([0,1]).all() and not labels.duplicated(keys).any()
    merged=labels.merge(test[keys+["alert","margin"]],on=keys,how="left",validate="one_to_one")
    merged[merged.alert.isna()].to_csv(RES/"unmatched_review_labels.csv",index=False)
    reviewed=merged.dropna(subset=["alert"])
    if len(reviewed):
        real=pd.DataFrame([metrics(reviewed.label,reviewed.alert.astype(int),reviewed.margin)])
        real["coverage_of_test"]=len(reviewed)/len(test)
        display(real); real.to_csv(RES/"reviewed_real_metrics.csv",index=False)
    else: print("Không có nhãn khớp cửa sổ test tháng 4.")
else:
    print("Accuracy trên bất thường thực tế: chưa có nhãn rà soát.")
print("Kết quả đã lưu tại",RES)

backup(RES)
